# Cross Validation

## Objective

This notebook demonstrates how cross-validation can provide a more reliable estimate of model performance than a single train/test split. We will be comparing the model performances of 5-fold cross-validation to single train/test split from the 03_bias_variance_tradeoff notebook. 

## Key Ideas

When dataset is split into training data to train the model, and testing data to evaluate the model, results may vary depending on getting an "easy" or "hard" test group. 
Cross-validation repeats this splitting multiple times to get a reliable final score for model performance.

Every data point gets a chance to be in both the training and testing datasets, thus preventing overfitting and produces a fairer evaluation of model performance.

## 01. Why Cross-Validation

A single train/test split evaluates a model on only a particular subset of data. The resulting performance can therefore be dependent on how the data happened to be split.

Cross-validation repeatedly splits the data into training and testing sets, allowing model performance to be evaluated across multiple splits.

## 02. K-Fold Cross-Validation

In this notebook, we will use the 5-fold cross-validation. The dataset will be split into 5 parts, with each part being used for testing once, and training 4 times. 

### 2.1 Generate Dataset

We will be generating the same synthetic dataset of age and systolic blood pressure from the 03_bias_variance_tradeoff notebook. 

In [2]:
#import modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
print("Modules imported")

#random number generator from numpy to create reproducible noise
np.random.seed(42)
#create 30 ages evenly (linearly) spaced between 20 to 80.
age = np.linspace(20, 80, 30)

#Synthetic systolic blood pressure with nonlinear r/s
blood_pressure = (100 + 0.4*age + 0.015 * (age-50)**2 + np.random.normal(0, 5, 30))

#Generate data into df of age and systolic bp
df = pd.DataFrame({'age': age, 'systolic_bp': blood_pressure})
df.head()

Modules imported


,age,systolic_bp
0,20.000000,123.983571
1,22.068966,119.838405
2,24.137931,122.926314
3,26.206897,126.589584
4,28.275862,117.218650


### 2.2 Apply Cross-Validation

In [ ]:
X = df[['age']]
y = df['systolic_bp']
degrees = [1, 2, 5]

#5-fold (split=5) cross-validation, shuffle the order (as age is currently ordered in data) reproducibly (randomstate)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for degree in degrees:
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    #use model, with X and y, use kf (5-fold cv) as defined earlier
    #sklearn scoring system returns negative MSE, hence x-1 to get ordinary MSE
    scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')
    mse = -scores
    #Return mean mse (average performance) and std dev mse (consistency of performance)
    results.append({'Degree': degree, 'Mean CV MSE': mse.mean(), 'Std CV MSE': mse.std()})

cv_results = pd.DataFrame(results)
print(cv_results)    

   Degree  Mean CV MSE  Std CV MSE
0       1    49.744913   11.089907
1       2    18.158776   12.005375
2       5    17.497265   10.182584


## 03. Compare Models

Cross-validation provides multiple estimates of model performance rather than relying on a single train/test split.

The mean cross-validation MSE provides an estimate of how well each model is expected to perform on unseen data, while the standard deviation shows how much performance varies between folds.

A lower mean MSE indicates better average performance.

Degree 1 model clearly performs worst, with high mean CV MSE supporting our ealier conclusion that the model is too simple and underfits.

Degree 2 model performs much better, reducing the mean CV MSE substantially. 

Degree 5 model has the lowest mean CV MSE. Among the 3 models, it has the best average validation performance. It also has the lowest standard deviation, meaning its performance was slightly more consistent across the five folds. 

In the previous single train/test split, degree 2 model appeared to be better fit model than the degree 5 model. With cross validation, degree 5 comes out slightly better.

## 04. Key takeaways

- A single train/test split can give an unreliable estimate of model performance depending on how the data is divided.
- K-fold cross-validation evaluates a model across multiple training/validation splits.
- Mean cross-validation performance provides a more reliable estimate of generalisation.
- Cross-validation can be used to compare models and help select an appropriate level of model complexity.